# Building a Chess Engine in Python

In this workshop, we'll build a working chess engine from scratch (well, almost — we'll lean on the `python-chess` library for the rules). By the end, you'll have an AI that can play a full game of chess against you.

**What we'll cover:**
1. Board representation with `python-chess`
2. Evaluating a chess position (how good is this board?)
3. The Minimax search algorithm
4. Alpha-Beta pruning (making it fast enough to be usable)
5. Putting it all together — playing a game!

**Prerequisites:** Basic Python knowledge. No chess expertise required (though it helps).

---
## 0. Setup

First, let's install and import what we need. We use `python-chess` — it handles all the chess rules (legal moves, check, checkmate, etc.) so we can focus on the *engine* logic.

In [ ]:
!pip install python-chess

In [ ]:
import chess
import math
from IPython.display import display, SVG

---
## 1. Board Representation

The `python-chess` library gives us a `Board` object that knows all the rules of chess. Let's explore it.

In [ ]:
# Create a board in the standard starting position
board = chess.Board()
board  # Jupyter renders this as an SVG board automatically

In [ ]:
# We can also see a text representation
print(board)

In [ ]:
# What legal moves are available?
print("Number of legal moves:", board.legal_moves.count())
print("Legal moves:", list(board.legal_moves))

In [ ]:
# Making and undoing moves
board.push(chess.Move.from_uci("e2e4"))  # Push a move (King's pawn opening)
print("After 1. e4:")
display(board)

board.pop()  # Undo the move
print("\nAfter undoing:")
display(board)

In [ ]:
# Looking at individual squares and pieces
piece = board.piece_at(chess.E2)  # What's on e2?
print(f"Piece on e2: {piece}")           # P = white Pawn
print(f"Piece type: {piece.piece_type}")  # 1 = PAWN
print(f"Piece color: {piece.color}")      # True = WHITE
print()

# Useful constants
print("chess.PAWN =", chess.PAWN)
print("chess.KNIGHT =", chess.KNIGHT)
print("chess.BISHOP =", chess.BISHOP)
print("chess.ROOK =", chess.ROOK)
print("chess.QUEEN =", chess.QUEEN)
print("chess.KING =", chess.KING)

In [ ]:
# The board knows about game state
print("Is check?", board.is_check())
print("Is checkmate?", board.is_checkmate())
print("Is game over?", board.is_game_over())
print("Whose turn?", "White" if board.turn == chess.WHITE else "Black")

### Quick Exercise

Create a board, play the **Scholar's Mate** (4-move checkmate), and verify the board reports checkmate.

The moves are: `e2e4`, `e7e5`, `f1c4`, `b8c6`, `d1h5`, `g8f6`, `h5f7`

In [ ]:
# Try it here!
exercise_board = chess.Board()

# Push the moves one by one...
# exercise_board.push(chess.Move.from_uci("e2e4"))
# ...

# Check: exercise_board.is_checkmate() should be True

---
## 2. Evaluating a Position

The heart of a chess engine is its **evaluation function**: given a board, how good is the position?

We'll score a position using two ideas:

### 2a. Material Value
Each piece has a point value. More material = better position.

| Piece  | Value |
|--------|-------|
| Pawn   | 100   |
| Knight | 320   |
| Bishop | 330   |
| Rook   | 500   |
| Queen  | 900   |
| King   | 20000 |

### 2b. Piece-Square Tables
Where a piece sits matters. A knight in the center is worth more than a knight in the corner. We encode this with **piece-square tables** — a bonus/penalty for each square, per piece type.

In [ ]:
# Piece values
piece_vals = {
    chess.PAWN: 100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK: 500,
    chess.QUEEN: 900,
    chess.KING: 20000
}

Piece-square tables are 64-element lists (one value per square). They're written from **Black's perspective** — rank 8 at the top, rank 1 at the bottom. For White, we reverse the table.

Here's how to read a pawn table — higher numbers mean the square is better for a pawn:
```
     a    b    c    d    d    e    f    g    h
8 [  0,   0,   0,   0,   0,   0,   0,   0 ]  <- promotion rank (won't stay here)
7 [ 50,  50,  50,  50,  50,  50,  50,  50 ]  <- almost promoting! big bonus
6 [ 10,  10,  20,  30,  30,  20,  10,  10 ]  <- center control rewarded
5 [  5,   5,  10,  25,  25,  10,   5,   5 ]
4 [  0,   0,   0,  20,  20,   0,   0,   0 ]  <- d4/e4 center pawns
3 [  5,  -5, -10,   0,   0, -10,  -5,   5 ]
2 [  5,  10,  10, -20, -20,  10,  10,   5 ]  <- discourages blocking bishops
1 [  0,   0,   0,   0,   0,   0,   0,   0 ]  <- starting rank
```

In [ ]:
# Piece-square tables (from Black's perspective, top = rank 8)
piece_table = {
    chess.PAWN: [
         0,  0,  0,  0,  0,  0,  0,  0,
        50, 50, 50, 50, 50, 50, 50, 50,
        10, 10, 20, 30, 30, 20, 10, 10,
         5,  5, 10, 25, 25, 10,  5,  5,
         0,  0,  0, 20, 20,  0,  0,  0,
         5, -5,-10,  0,  0,-10, -5,  5,
         5, 10, 10,-20,-20, 10, 10,  5,
         0,  0,  0,  0,  0,  0,  0,  0
    ],
    chess.KNIGHT: [
        -50,-40,-30,-30,-30,-30,-40,-50,
        -40,-20,  0,  0,  0,  0,-20,-40,
        -30,  0, 10, 15, 15, 10,  0,-30,
        -30,  5, 15, 20, 20, 15,  5,-30,
        -30,  0, 15, 20, 20, 15,  0,-30,
        -30,  5, 10, 15, 15, 10,  5,-30,
        -40,-20,  0,  5,  5,  0,-20,-40,
        -50,-40,-30,-30,-30,-30,-40,-50,
    ],
    chess.BISHOP: [
        -20,-10,-10,-10,-10,-10,-10,-20,
        -10,  0,  0,  0,  0,  0,  0,-10,
        -10,  0,  5, 10, 10,  5,  0,-10,
        -10,  5,  5, 10, 10,  5,  5,-10,
        -10,  0, 10, 10, 10, 10,  0,-10,
        -10, 10, 10, 10, 10, 10, 10,-10,
        -10,  5,  0,  0,  0,  0,  5,-10,
        -20,-10,-10,-10,-10,-10,-10,-20,
    ],
    chess.ROOK: [
         0,  0,  0,  0,  0,  0,  0,  0,
         5, 10, 10, 10, 10, 10, 10,  5,
        -5,  0,  0,  0,  0,  0,  0, -5,
        -5,  0,  0,  0,  0,  0,  0, -5,
        -5,  0,  0,  0,  0,  0,  0, -5,
        -5,  0,  0,  0,  0,  0,  0, -5,
        -5,  0,  0,  0,  0,  0,  0, -5,
         0,  0,  0,  5,  5,  0,  0,  0,
    ],
    chess.QUEEN: [
        -20,-10,-10, -5, -5,-10,-10,-20,
        -10,  0,  0,  0,  0,  0,  0,-10,
        -10,  0,  5,  5,  5,  5,  0,-10,
         -5,  0,  5,  5,  5,  5,  0, -5,
          0,  0,  5,  5,  5,  5,  0, -5,
        -10,  5,  5,  5,  5,  5,  0,-10,
        -10,  0,  5,  0,  0,  0,  0,-10,
        -20,-10,-10, -5, -5,-10,-10,-20,
    ],
    chess.KING: [
        -30,-40,-40,-50,-50,-40,-40,-30,
        -30,-40,-40,-50,-50,-40,-40,-30,
        -30,-40,-40,-50,-50,-40,-40,-30,
        -30,-40,-40,-50,-50,-40,-40,-30,
        -20,-30,-30,-40,-40,-30,-30,-20,
        -10,-20,-20,-20,-20,-20,-20,-10,
         20, 20,  0,  0,  0,  0, 20, 20,
         20, 30, 10,  0,  0, 10, 30, 20,
    ]
}

Now let's write the evaluation function. The idea is simple:

```
score = 0
for each piece on the board:
    if it's OUR piece:  score += piece_value + square_bonus
    if it's THEIR piece: score -= piece_value + square_bonus
```

A positive score means the position favors us; negative means it favors the opponent.

In [ ]:
def evaluate(board, is_white=False):
    """
    Evaluate the board from one side's perspective.
    
    Args:
        board: a chess.Board object
        is_white: if True, evaluate from White's perspective; otherwise Black's
    
    Returns:
        An integer score. Positive = good for us, negative = good for opponent.
    """
    score = 0

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is None:
            continue

        value = piece_vals[piece.piece_type]
        table = piece_table[piece.piece_type]

        if is_white:
            if piece.color == chess.WHITE:
                # Tables are from Black's view, so reverse for White
                score += value + table[::-1][square]
            else:
                score -= value + table[square]
        else:
            if piece.color == chess.BLACK:
                score += value + table[square]
            else:
                # Reverse table for the opponent (White)
                score -= value + table[::-1][square]

    return score

In [ ]:
# Let's test it on the starting position — should be roughly 0 (symmetric)
board = chess.Board()
print("Starting position (White's view):", evaluate(board, is_white=True))
print("Starting position (Black's view):", evaluate(board, is_white=False))

In [ ]:
# Now let's remove Black's queen and see the difference
board_no_queen = chess.Board("rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
print("Without Black's queen (White's view):", evaluate(board_no_queen, is_white=True))
display(board_no_queen)

### Quick Exercise

Set up a board where White is up a rook (remove Black's rook from a8). What does `evaluate()` return for White? Does the number make sense given the piece values?

In [ ]:
# Try it here! Hint: use chess.Board("FEN string") with the a8 rook removed.


---
## 3. The Minimax Algorithm

We can evaluate a *single* position. But chess is about looking *ahead*. If I take your queen but you checkmate me next move, that was a bad trade.

**Minimax** is the core idea: assume both players play optimally.
- On **our** turn, we pick the move that **maximizes** our score.
- On **their** turn, they pick the move that **minimizes** our score.
- We alternate, looking several moves ahead (the **depth**).

```
                     Current Position
                    /       |        \
               Move A    Move B    Move C      <- WE maximize
              /     \    /    \    /     \
           D1  D2  D3  D4   D5  D6  D7  D8   <- THEY minimize
           |   |   |   |    |   |   |    |
          ...evaluate leaf positions...        <- evaluate()
```

In [ ]:
def minimax(board, depth, is_maximizing, is_white):
    """
    Minimax search (no pruning yet — we'll add that next).
    
    Args:
        board: current chess.Board
        depth: how many more levels to search
        is_maximizing: True if it's the maximizing player's turn
        is_white: perspective for evaluation
    
    Returns:
        The best score reachable from this position.
    """
    # Base cases: stop searching
    if board.is_checkmate():
        return 100000  # Checkmate is the best possible outcome
    if depth == 0 or board.is_game_over():
        return evaluate(board, is_white)

    if is_maximizing:
        best_score = -math.inf
        for move in board.legal_moves:
            board.push(move)
            score = minimax(board, depth - 1, False, is_white)
            board.pop()
            best_score = max(best_score, score)
        return best_score

    else:  # minimizing
        best_score = math.inf
        for move in board.legal_moves:
            board.push(move)
            score = minimax(board, depth - 1, True, is_white)
            board.pop()
            best_score = min(best_score, score)
        return best_score

### The problem: it's slow

Chess has an average of ~35 legal moves per position. Searching to depth 3 means evaluating roughly **35 x 35 x 35 = ~42,000** positions. Depth 4 is ~1.5 million. We need to do better.

---
## 4. Alpha-Beta Pruning

**Key insight:** we can skip entire branches of the search tree when we *know* they can't affect the result.

Imagine the maximizer has already found a move scoring **50**. Now they're examining another branch, and the minimizer finds a reply scoring **30**. The minimizer might find even *lower* scores in that branch — so the maximizer will never choose it (they already have 50). We can **prune** (skip) the rest of that branch.

We track two values:
- **alpha**: the best score the maximizer can guarantee so far
- **beta**: the best score the minimizer can guarantee so far
- When **beta <= alpha**, we prune (stop searching this branch)

In [ ]:
def minimax_ab(board, depth, alpha, beta, is_maximizing, is_white):
    """
    Minimax with alpha-beta pruning.
    Same logic as before, but we skip branches that can't change the outcome.
    """
    if board.is_checkmate():
        return 100000
    if depth == 0 or board.is_game_over():
        return evaluate(board, is_white)

    if is_maximizing:
        best_score = -math.inf
        for move in board.legal_moves:
            board.push(move)
            score = minimax_ab(board, depth - 1, alpha, beta, False, is_white)
            board.pop()
            best_score = max(best_score, score)
            alpha = max(alpha, score)
            if beta <= alpha:
                break  # Beta cutoff — prune!
        return best_score

    else:
        best_score = math.inf
        for move in board.legal_moves:
            board.push(move)
            score = minimax_ab(board, depth - 1, alpha, beta, True, is_white)
            board.pop()
            best_score = min(best_score, score)
            beta = min(beta, score)
            if beta <= alpha:
                break  # Alpha cutoff — prune!
        return best_score

### Speed comparison

Let's see how much faster alpha-beta pruning makes our search.

In [ ]:
import time

board = chess.Board()

# Without pruning
start = time.time()
score_no_prune = minimax(board, 3, True, True)
time_no_prune = time.time() - start

# With pruning
start = time.time()
score_pruned = minimax_ab(board, 3, -math.inf, math.inf, True, True)
time_pruned = time.time() - start

print(f"Without pruning: {time_no_prune:.2f}s (score: {score_no_prune})")
print(f"With pruning:    {time_pruned:.2f}s (score: {score_pruned})")
print(f"Speedup:         {time_no_prune / time_pruned:.1f}x")

Same result, much faster. Alpha-beta pruning typically lets you search **~twice as deep** in the same time.

---
## 5. Finding the Best Move

Now we need a function that actually *picks* a move. It tries every legal move, scores each one using `minimax_ab`, and returns the best.

In [ ]:
def find_move(board, depth, is_white=False):
    """
    Find the best move for the current side.
    
    Args:
        board: current chess.Board
        depth: search depth
        is_white: True if finding a move for White
    
    Returns:
        The best chess.Move found.
    """
    best_move = None
    best_value = -math.inf

    for move in board.legal_moves:
        board.push(move)
        move_value = minimax_ab(board, depth - 1, -math.inf, math.inf, False, is_white)
        board.pop()

        if move_value > best_value:
            best_value = move_value
            best_move = move

    return best_move

In [ ]:
# Let's see what the engine thinks is the best opening move for White
board = chess.Board()
best = find_move(board, depth=3, is_white=True)
print(f"Best opening move (depth 3): {best}")

---
## 6. Let's Play!

Now we put it all together. The engine will play against itself so you can watch it in action.

In [ ]:
def play_game(depth=3, max_moves=50):
    """
    Have the engine play against itself.
    
    Args:
        depth: search depth for both sides
        max_moves: stop after this many full moves (to keep the demo short)
    """
    board = chess.Board()
    move_count = 0

    while not board.is_game_over() and move_count < max_moves:
        # White's turn
        white_move = find_move(board, depth, is_white=True)
        if white_move is None:
            break
        board.push(white_move)
        if board.is_game_over():
            break

        # Black's turn
        black_move = find_move(board, depth, is_white=False)
        if black_move is None:
            break
        board.push(black_move)

        move_count += 1

    # Show the final position
    display(board)
    if board.is_game_over():
        print(f"Game over after {move_count} full moves: {board.outcome()}")
    else:
        print(f"Stopped after {move_count} full moves (game still in progress).")
    print(f"White's eval: {evaluate(board, is_white=True)}")
    return board

In [ ]:
# Watch the engine play against itself (this may take a minute or two)
final_board = play_game(depth=3, max_moves=30)

---
## 7. Play Against the Engine!

Now it's your turn. You play as White, the engine plays as Black.

In [ ]:
def play_vs_engine(engine_depth=3):
    """
    Play against the engine. You are White, the engine is Black.
    Type moves in UCI format (e.g. 'e2e4', 'g1f3', 'e1g1' for kingside castling).
    Type 'quit' to stop.
    """
    board = chess.Board()

    while not board.is_game_over():
        display(board)

        # --- Your move (White) ---
        move = None
        while move is None:
            uci = input("Your move (UCI format, e.g. 'e2e4'): ").strip()
            if uci == "quit":
                print("Thanks for playing!")
                return board
            try:
                move = chess.Move.from_uci(uci)
                if move not in board.legal_moves:
                    print(f"Illegal move: {uci}. Legal moves: {[m.uci() for m in board.legal_moves]}")
                    move = None
            except ValueError:
                print(f"Invalid format: {uci}. Use UCI notation like 'e2e4'.")

        board.push(move)
        if board.is_game_over():
            break

        # --- Engine's move (Black) ---
        print("Engine is thinking...")
        engine_move = find_move(board, engine_depth, is_white=False)
        print(f"Engine plays: {engine_move}")
        board.push(engine_move)

    display(board)
    print(f"Game over! {board.outcome()}")
    return board

In [ ]:
# Uncomment to play! (comment it back out before running the whole notebook)
# play_vs_engine(engine_depth=3)

---
## 8. Ideas to Explore Further

Our engine works, but there's so much more you could add! Here are some ideas, roughly in order of complexity:

1. **Increase search depth** — Try depth 4 or 5. How much slower is it? How much better does it play?

2. **Move ordering** — Alpha-beta pruning works best when good moves are searched first. Try sorting captures before quiet moves.

3. **Quiescence search** — At depth 0, instead of just evaluating, keep searching capture moves to avoid the [horizon effect](https://www.chessprogramming.org/Horizon_Effect).

4. **Better evaluation** — Add bonuses for:
   - Passed pawns (no enemy pawns blocking them)
   - King safety (pawn shield)
   - Rooks on open files
   - Bishop pair bonus

5. **Iterative deepening** — Search depth 1, then 2, then 3... Use the result of shallower searches to order moves for deeper searches.

6. **Transposition table** — Cache evaluated positions (many positions can be reached by different move orders).

7. **Opening book** — Use a database of known good openings for the first few moves.

Happy hacking!